**ChatGPT-4.1-mini(API)를 활용한 행위자 관점 분석**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 패키지 설치 (Jupyter/Colab에서 실행)
# ═══════════════════════════════════════════════════════════════════════
"""
!pip install pandas openai tqdm nest-asyncio
"""

# ═══════════════════════════════════════════════════════════════════════
# 행위자 중심 관점 분석기 (Actor-Centered Tone Analysis)
# ═══════════════════════════════════════════════════════════════════════

import pandas as pd
from openai import AsyncOpenAI
import asyncio
import json
import os
import logging
import time
from typing import Tuple, Dict
import re
from tqdm import tqdm

# ═══════════════════════════════════════════════════════════════════════
# 1. 관점(Tone) 정의
# ═══════════════════════════════════════════════════════════════════════
ALLOWED_TONES = {
    "긍정": {
        "description": "기사가 [분석 대상 행위자]를 긍정적으로 보도하는 경우. 업적, 성과, 긍정적 평가, 지지, 호의적 묘사 등이 주를 이룸"
    },
    "중립": {
        "description": "단순 사실 전달 또는 긍정과 부정이 혼재된 경우. 다만 맥락상 긍정이나 부정이 조금 더 많다면 해당 방향으로 판단. 객관적 보도, 양측 의견 균형적 제시"
    },
    "부정": {
        "description": "기사가 [분석 대상 행위자]를 부정적으로 보도하는 경우. 비판, 문제 제기, 부정적 평가, 반대, 비호의적 묘사 등이 주를 이룸"
    }
}

def format_allowed_tones():
    """관점을 프롬프트용 문자열로 변환"""
    tone_list = []
    for tone_name, tone_info in ALLOWED_TONES.items():
        tone_text = f"- {tone_name}: {tone_info['description']}"
        tone_list.append(tone_text)
    return "\n".join(tone_list)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2. 프롬프트 템플릿
# ═══════════════════════════════════════════════════════════════════════
PROMPT_TEMPLATE = """다음 뉴스 기사에서 "[분석 대상 행위자]"에 대한 관점(tone)을 분석해주세요.

[뉴스 기사]:
{text}

[관점 분류 기준]:
{allowed_tones}

**분석 방법:**
1. 기사에서 [분석 대상 행위자]가 어떻게 묘사되고 평가되는지 파악하세요
2. 긍정적 표현, 부정적 표현, 중립적 서술의 비중을 고려하세요
3. 맥락상 긍정이나 부정 중 하나가 조금 더 우세하다면 해당 방향으로 판단하세요
4. 위 3가지 관점 중 하나를 선택하세요
5. 선택한 관점에 대한 구체적 근거를 제시하세요
6. 분류 신뢰도를 1-100 사이의 정수로 평가하세요

**반드시 다음 JSON 형식으로만 답변하세요:**
{{
  "tone": "긍정 또는 중립 또는 부정",
  "reason": "해당 관점으로 분류한 구체적 근거 (기사 내 표현과 맥락 언급)",
  "confidence": 85
}}

답변 주의사항:
- tone 필드는 "긍정", "중립", "부정" 중 정확히 하나만 선택
- reason 필드는 기사 내용을 근거로 구체적으로 작성
- confidence는 1-100 사이의 정수
- JSON 외 다른 내용 금지

JSON 응답:"""

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3. JSON 파서
# ═══════════════════════════════════════════════════════════════════════
class EnhancedJSONParser:
    """강화된 JSON 파싱 클래스"""
    
    @staticmethod
    def extract_json_robust(response_text: str) -> Dict:
        """강화된 JSON 추출 로직"""
        if not response_text or not response_text.strip():
            return EnhancedJSONParser._get_default_result("빈 응답")
        
        response_text = response_text.strip()
        
        # 1단계: 직접 JSON 파싱 시도
        try:
            if response_text.startswith('{') and response_text.endswith('}'):
                return json.loads(response_text)
        except json.JSONDecodeError:
            pass
        
        # 2단계: 마크다운 JSON 블록 추출
        json_patterns = [
            r'```json\s*\n?(.*?)\n?```',
            r'```\s*\n?(.*?)\n?```',
            r'`{3,}\s*json\s*\n?(.*?)\n?`{3,}',
        ]
        
        for pattern in json_patterns:
            match = re.search(pattern, response_text, re.DOTALL | re.IGNORECASE)
            if match:
                json_str = match.group(1).strip()
                try:
                    return json.loads(json_str)
                except json.JSONDecodeError:
                    continue
        
        # 3단계: 중괄호 블록 추출
        brace_patterns = [
            r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}',
            r'\{(?:[^{}]|{[^{}]*})*\}',
        ]
        
        for pattern in brace_patterns:
            matches = re.findall(pattern, response_text)
            for match in matches:
                try:
                    result = json.loads(match)
                    if EnhancedJSONParser._validate_result(result):
                        return result
                except json.JSONDecodeError:
                    continue
        
        # 4단계: 키-값 개별 추출
        extracted_data = {}
        
        # tone 필드 추출
        tone_patterns = [
            r'"tone"\s*:\s*"([^"]+)"',
            r'tone["\']?\s*[:=]\s*["\']([^"\']+)["\']',
        ]
        
        for pattern in tone_patterns:
            match = re.search(pattern, response_text, re.IGNORECASE)
            if match:
                extracted_data['tone'] = match.group(1).strip()
                break
        
        # reason 필드 추출
        reason_patterns = [
            r'"reason"\s*:\s*"([^"]*(?:\\.[^"]*)*)"',
            r'reason["\']?\s*[:=]\s*["\']([^"\']+)["\']',
        ]
        
        for pattern in reason_patterns:
            match = re.search(pattern, response_text, re.IGNORECASE | re.DOTALL)
            if match:
                extracted_data['reason'] = match.group(1).strip()
                break
        
        # confidence 필드 추출
        confidence_patterns = [
            r'"confidence"\s*:\s*([0-9]+)',
            r'confidence["\']?\s*[:=]\s*([0-9]+)',
        ]
        
        for pattern in confidence_patterns:
            match = re.search(pattern, response_text, re.IGNORECASE)
            if match:
                try:
                    extracted_data['confidence'] = int(match.group(1))
                except ValueError:
                    pass
                break
        
        if extracted_data:
            result = {
                'tone': extracted_data.get('tone', '분류 실패'),
                'reason': extracted_data.get('reason', '근거 추출 실패'),
                'confidence': extracted_data.get('confidence', 0)
            }
            return result
        
        # 5단계: 기본값 반환
        return EnhancedJSONParser._get_default_result("파싱 실패")
    
    @staticmethod
    def _validate_result(result: Dict) -> bool:
        """결과 유효성 검증"""
        required_keys = ['tone', 'reason', 'confidence']
        return all(key in result for key in required_keys)
    
    @staticmethod
    def _get_default_result(reason: str) -> Dict:
        """기본 결과 반환"""
        return {
            'tone': '분류 실패',
            'reason': f'JSON 파싱 실패: {reason}',
            'confidence': 0
        }

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4. 관점 분석기
# ═══════════════════════════════════════════════════════════════════════
class ToneAnalyzer:
    """행위자 중심 관점 분석기"""
    
    def __init__(self, api_key, model, output_dir='.', 
                 concurrent_requests=3, rate_limit_delay=1.0, batch_size=5):
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model
        self.output_dir = output_dir
        self.concurrent_requests = concurrent_requests
        self.rate_limit_delay = rate_limit_delay
        self.batch_size = batch_size
        self.total_cost = 0.0
        self.semaphore = asyncio.Semaphore(concurrent_requests)
        self.json_parser = EnhancedJSONParser()
        
        # 통계
        self.parsing_successes = 0
        self.parsing_failures = 0
        
        # 로깅 설정 (WARNING 레벨로 최소화)
        logging.basicConfig(level=logging.WARNING, format='%(message)s')
        self.logger = logging.getLogger(__name__)
        
        os.makedirs(self.output_dir, exist_ok=True)
        
    def combine_title_text(self, title: str, text: str) -> str:
        """제목과 본문을 결합"""
        if pd.isna(title):
            title = ""
        if pd.isna(text):
            text = ""
        
        title = str(title).strip()
        text = str(text).strip()
        
        combined = f"{title}\n\n{text}" if title and text else (title or text)
        return combined[:4000]  # 토큰 제한을 위한 길이 제한

    def normalize_tone(self, tone: str) -> str:
        """관점명 정규화 - 정확한 매칭만 사용"""
        if not tone or tone == '분류 실패':
            return "분류 실패"
        
        tone = tone.strip()
        
        # 정확한 매칭만 허용
        if tone in ALLOWED_TONES:
            return tone
        
        return "분류 실패"

    async def analyze_single_article(self, title: str, text: str) -> Tuple[str, str, int, float]:
        """단일 기사 분석 - 각 호출은 완전히 독립적"""
        
        async with self.semaphore:
            try:
                # 제목과 본문 결합
                combined_text = self.combine_title_text(title, text)
                
                if not combined_text or combined_text == "":
                    return "분류 실패", "빈 텍스트", 0, 0.0
                
                # 프롬프트 생성 (매번 독립적으로 생성)
                prompt = PROMPT_TEMPLATE.format(
                    text=combined_text,
                    allowed_tones=format_allowed_tones()
                )
                
                # API 호출 (각 호출은 독립적인 세션)
                response = await self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "user", "content": prompt}
                    ],
                    max_tokens=500,
                    temperature=0,
                    response_format={"type": "json_object"}
                )
                
                response_text = response.choices[0].message.content.strip()
                result = self.json_parser.extract_json_robust(response_text)
                
                # 파싱 성공/실패 통계
                if result.get('tone') == '분류 실패':
                    self.parsing_failures += 1
                else:
                    self.parsing_successes += 1
                
                # 결과 추출
                tone = self.normalize_tone(result.get("tone", "분류 실패"))
                reason = result.get("reason", "근거 없음").replace('"reason":', '').replace('"', '').strip()
                confidence = int(result.get("confidence", 0))
                
                # 신뢰도 범위 검증
                confidence = max(0, min(100, confidence))
                
                # 비용 계산
                estimated_cost = 0.002
                self.total_cost += estimated_cost
                
                return tone, reason, confidence, estimated_cost
                
            except Exception as e:
                self.parsing_failures += 1
                return "분류 실패", f"오류: {str(e)[:50]}", 0, 0.0
            
            finally:
                await asyncio.sleep(self.rate_limit_delay)

    async def process_batch(self, batch_df) -> Tuple[list, list, list, list]:
        """배치 처리"""
        tasks = []
        
        for _, row in batch_df.iterrows():
            task = self.analyze_single_article(
                row['title'], 
                row['text']
            )
            tasks.append(task)
        
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        tones = []
        reasons = []
        confidences = []
        costs = []
        
        for result in results:
            if isinstance(result, Exception):
                tones.append("분류 실패")
                reasons.append(f"배치 오류: {str(result)[:50]}")
                confidences.append(0)
                costs.append(0.0)
                self.parsing_failures += 1
            else:
                tones.append(result[0])
                reasons.append(result[1])
                confidences.append(result[2])
                costs.append(result[3])
        
        return tones, reasons, confidences, costs

    async def analyze_dataframe(self, df, output_filename="tone_analysis_results.csv"):
        """전체 데이터프레임 분석"""
        
        output_path = os.path.join(self.output_dir, output_filename)
        
        all_tones = []
        all_reasons = []
        all_confidences = []
        all_costs = []
        
        start_time = time.time()
        
        # tqdm 진행바 생성
        pbar = tqdm(total=len(df), desc="관점 분석 중", unit="건")
        
        # 배치별 처리
        for i in range(0, len(df), self.batch_size):
            batch_end = min(i + self.batch_size, len(df))
            batch_df = df.iloc[i:batch_end]
            
            try:
                tones, reasons, confidences, costs = await self.process_batch(batch_df)
                
                all_tones.extend(tones)
                all_reasons.extend(reasons)
                all_confidences.extend(confidences)
                all_costs.extend(costs)
                
                # 진행바 업데이트
                pbar.update(len(batch_df))
                
                # 중간 저장 (5배치마다)
                batch_num = i // self.batch_size + 1
                if batch_num % 5 == 0:
                    temp_df = df.copy()
                    temp_df['관점'] = all_tones + [''] * (len(df) - len(all_tones))
                    temp_df['근거'] = all_reasons + [''] * (len(df) - len(all_reasons))
                    temp_df['신뢰도'] = all_confidences + [0] * (len(df) - len(all_confidences))
                    temp_df['비용'] = all_costs + [0.0] * (len(df) - len(all_costs))
                    temp_df.to_csv(output_path, encoding='utf-8-sig', index=False)
                
            except Exception as e:
                batch_size = len(batch_df)
                all_tones.extend(["분류 실패"] * batch_size)
                all_reasons.extend([f"배치 오류: {str(e)[:30]}"] * batch_size)
                all_confidences.extend([0] * batch_size)
                all_costs.extend([0.0] * batch_size)
                self.parsing_failures += batch_size
                pbar.update(len(batch_df))
        
        pbar.close()
        
        # 최종 결과 생성
        result_df = df.copy()
        result_df['관점'] = all_tones
        result_df['근거'] = all_reasons
        result_df['신뢰도'] = all_confidences
        result_df['비용'] = all_costs
        
        # 최종 저장
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)
        
        # 완료 메시지
        elapsed_time = time.time() - start_time
        self._print_completion_message(result_df, elapsed_time, output_path)
        
        return result_df

    def _print_completion_message(self, result_df, elapsed_time, output_path):
        """완료 메시지 출력"""
        total_count = len(result_df)
        tone_distribution = result_df['관점'].value_counts()
        
        print(f"\n{'='*60}")
        print(f"✅ 행위자 관점 분석 완료!")
        print(f"{'='*60}")
        print(f"📊 총 {total_count:,}건 뉴스 분석 완료 (소요시간: {elapsed_time/60:.1f}분)")
        print(f"💰 총 비용: ${self.total_cost:.4f}")
        print(f"📁 결과 파일: {output_path}")
        print(f"\n📈 관점 분포:")
        for tone, count in tone_distribution.items():
            percentage = (count / total_count) * 100
            print(f"   • {tone}: {count:,}건 ({percentage:.1f}%)")
        print(f"{'='*60}\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 5. 설정 및 실행
# ═══════════════════════════════════════════════════════════════════════
# ──────────────────────────────────────────────────────────────────────
# 여기서 설정을 변경하세요
# ──────────────────────────────────────────────────────────────────────
MODEL = "gpt-4.1-mini"  # "gpt-4o-mini" 또는 "gpt-4o"
API_KEY = "Your-OpenAI-API-Key-Here"
INPUT_CSV_PATH = "Input CSV file path here"
OUTPUT_CSV_PATH = "output CSV file path here"
# ──────────────────────────────────────────────────────────────────────

async def main():
    """메인 함수"""
    try:
        # CSV 파일 로드
        df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8-sig')
        
        # 필수 컬럼 확인
        if 'title' not in df.columns or 'text' not in df.columns:
            print("❌ title, text 컬럼 필요")
            return
        
        # 출력 경로 설정
        output_dir = os.path.dirname(OUTPUT_CSV_PATH) or '.'
        output_file = os.path.basename(OUTPUT_CSV_PATH)
        
        # 분석기 생성 및 실행
        analyzer = ToneAnalyzer(
            api_key=API_KEY,
            model=MODEL,
            output_dir=output_dir
        )
        
        await analyzer.analyze_dataframe(df, output_file)
        
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없음: {INPUT_CSV_PATH}")
    except Exception as e:
        print(f"❌ 오류: {str(e)}")

def run_analysis():
    """분석 실행"""
    try:
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.run(main())
    except ImportError:
        asyncio.run(main())

if __name__ == "__main__":
    run_analysis()